## Kub-vip as VIP for cluster internal

Install kube-vip as static pod

In [ ]:
sudo -i 

VIP="172.16.6.85"
INTERFACE="ens192"

KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases | jq -r ".[0].name")

alias kube-vip="ctr image pull ghcr.io/kube-vip/kube-vip:$KVVERSION; ctr run --rm --net-host ghcr.io/kube-vip/kube-vip:$KVVERSION vip /kube-vip"

mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface $INTERFACE \
    --address $VIP \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

In [ ]:
ctr images ls | grep kube-vip

Initialize the cluster

In [ ]:
kubeadm init   --control-plane-endpoint "172.16.6.85:6443"   --upload-certs --pod-network-cidr=10.244.0.0/16

In [ ]:
# If you need the join command
kubeadm init phase upload-certs --upload-certs

kubeadm token create --print-join-command

In [ ]:
export KUBECONFIG=/etc/kubernetes/admin.conf

In [ ]:
mkdir -p $HOME/.kube
sudo cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
sudo chown $(id -u):$(id -g) $HOME/.kube/config

---

> We have more one CNI to work with it

#### 1. Fannel (We will work wit it)

In [ ]:
kubectl apply -f https://github.com/flannel-io/flannel/releases/latest/download/kube-flannel.yml

# Wait for ready
kubectl rollout status daemonset kube-flannel-ds -n kube-flannel
watch kubectl get pods -n kube-flannel

`On all nodes`

In [ ]:
sudo dnf install containernetworking-plugins -y

sudo cp /usr/libexec/cni/* /opt/cni/bin/

sudo systemctl restart containerd
sudo systemctl restart kubelet

#### 2. Calico

In [ ]:
# 1. Install operator
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.29.3/manifests/tigera-operator.yaml
kubectl rollout status deployment tigera-operator -n tigera-operator

# 2. Apply with VXLAN - save this file
cat <<EOF > calico-custom-resources.yaml
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  calicoNetwork:
    ipPools:
    - name: default-ipv4-ippool
      cidr: 10.244.0.0/16
      encapsulation: VXLAN
      natOutgoing: Enabled
      nodeSelector: all()
---
apiVersion: operator.tigera.io/v1
kind: APIServer
metadata:
  name: default
spec: {}
EOF

kubectl create -f calico-custom-resources.yaml

# 3. Wait for everything to be ready
watch kubectl get pods -n calico-system
# All pods must show 1/1 Running before continuing

In [ ]:
kubectl run test-1 --image=nginx:alpine --overrides='{"spec":{"nodeName":"worker-1"}}'
kubectl run test-2 --image=nginx:alpine --overrides='{"spec":{"nodeName":"worker-2"}}'

# Wait for running
kubectl get pods -o wide

# Test cross-node
TEST2_IP=$(kubectl get pod test-2 -o jsonpath='{.status.podIP}')
kubectl exec test-1 -- curl -s --max-time 5 http://$TEST2_IP

# Must return nginx page before continuing
kubectl delete pod test-1 test-2

---

Old Data calico

In [ ]:
# 1. Download the operator
kubectl create -f https://raw.githubusercontent.com/projectcalico/calico/v3.29.3/manifests/tigera-operator.yaml

# 2. Create custom-resources.yaml with VXLAN
cat <<EOF > calico-custom-resources.yaml
apiVersion: operator.tigera.io/v1
kind: Installation
metadata:
  name: default
spec:
  cniPlugin: Calico
  calicoNetwork:
    ipPools:
    - name: default-ipv4-ippool
      cidr: 10.244.0.0/16
      encapsulation: VXLAN        # This is the key - works on any network
      natOutgoing: Enabled
      nodeSelector: all()
---
apiVersion: operator.tigera.io/v1
kind: APIServer
metadata:
  name: default
spec: {}
EOF

kubectl create -f calico-custom-resources.yaml

Copy the manifests to the other masters (Becouse we use static pod)

In [ ]:
scp /etc/kubernetes/manifests/kube-vip.yaml root@172.16.6.66:/etc/kubernetes/manifests/
scp /etc/kubernetes/manifests/kube-vip.yaml root@172.16.6.62:/etc/kubernetes/manifests/

---

Old data for kube-vip

In [ ]:

VIP="172.16.6.85"
INTERFACE="ens192"

KVVERSION=$(curl -sL https://api.github.com/repos/kube-vip/kube-vip/releases | jq -r ".[0].name")

alias kube-vip="ctr image pull ghcr.io/kube-vip/kube-vip:$KVVERSION; ctr run --rm --net-host ghcr.io/kube-vip/kube-vip:$KVVERSION vip /kube-vip"

mkdir -p /etc/kubernetes/manifests/

kube-vip manifest pod \
    --interface $INTERFACE \
    --address $VIP \
    --controlplane \
    --services \
    --arp \
    --leaderElection | tee /etc/kubernetes/manifests/kube-vip.yaml

ctr images ls | grep kube-vip

kubeadm init   --control-plane-endpoint "172.16.6.85:6443"   --upload-certs --pod-network-cidr=10.244.0.0/16



mkdir -p $HOME/.kube
sudo cp -i /etc/kubernetes/admin.conf $HOME/.kube/config
sudo chown $(id -u):$(id -g) $HOME/.kube/config


kubectl apply -f https://raw.githubusercontent.com/projectcalico/calico/v3.27.5/manifests/calico.yaml

# From master1, or copy the file manually:
scp /etc/kubernetes/manifests/kube-vip.yaml master2:/etc/kubernetes/manifests/
scp /etc/kubernetes/manifests/kube-vip.yaml master3:/etc/kubernetes/manifests/

# If we need

kubeadm init phase upload-certs --upload-certs

kubeadm token create --print-join-command

